# 📊 SuperStore Executive Dashboard — Data Pipeline
**Mục đích:** Đọc dữ liệu SuperStore, dịch thời gian +9 năm (2014-2017 → 2023-2026), tính tất cả tham số cho dashboard

**Công thức pace goal (mùa vụ):**
```
pace_goal = annual_goal × pace_weight
pace_weight = Σ weight(T1..T5) + weight(T6) × (5/30)
```
Trọng số từng tháng lấy từ tỷ trọng doanh thu thực tế năm 2025.
V2:
Trọng số các tháng năm nay lấy từ mô hình dự đoán

In [1]:
import pandas as pd
import numpy as np
import json
import calendar
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════
# 1. CẤU HÌNH — chỉnh sửa tại đây
# ═══════════════════════════════════════════════════════════════
CSV_PATH    = '/content/SampleSuperstore.csv'       # đường dẫn file dữ liệu
SHIFT_YEARS = 9                       # dịch năm: 2014+9=2023, 2017+9=2026
#Dữ liệu từ ngày hôm qua, báo cáo vào ngày hôm nay
CUTOFF = pd.Timestamp(datetime.today().date()) - pd.Timedelta(days=1)

# Mục tiêu tăng trưởng (%) — chỉnh sửa theo kế hoạch thực tế
with open('/content/kpi_parameter.json', encoding='utf-8') as f:
    _kpi_param = json.load(f)
GROWTH = _kpi_param['growth_targets']
PROFIT_MARGIN_TARGET = _kpi_param['profit_margin_target']
AOV_TARGET = _kpi_param['avg_order_value_target']

print('✅ Cấu hình nạp xong')

✅ Cấu hình nạp xong


In [2]:
# ═══════════════════════════════════════════════════════════════
# 2. ĐỌC & DỊCH DỮ LIỆU
# ═══════════════════════════════════════════════════════════════
df = pd.read_csv(CSV_PATH, encoding='latin1')
df['Order Date'] = pd.to_datetime(df['Order Date'])

def shift_date(dt, years=SHIFT_YEARS):
    """Dịch năm, xử lý ngày 29/2 leap year."""
    try:
        return dt.replace(year=dt.year + years)
    except ValueError:
        return dt.replace(year=dt.year + years, day=28)

df['Order Date'] = df['Order Date'].apply(shift_date)

YR_START   = pd.Timestamp(f'{CUTOFF.year}-01-01')
PREV_YEAR  = CUTOFF.year - 1
PREV_START = pd.Timestamp(f'{PREV_YEAR}-01-01')
PREV_END   = pd.Timestamp(f'{PREV_YEAR}-12-31')

# Lọc các kỳ cần thiết
df_curr     = df[(df['Order Date'] >= YR_START)  & (df['Order Date'] <= CUTOFF)].copy()
df_prev     = df[(df['Order Date'] >= PREV_START)& (df['Order Date'] <= PREV_END)].copy()
df_prev_ytd = df[(df['Order Date'] >= PREV_START)& (df['Order Date'] <= PREV_START + (CUTOFF - YR_START))].copy()

print(f'Phạm vi dữ liệu sau dịch: {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print(f'YTD {CUTOFF.year}: {len(df_curr):,} giao dịch | Doanh thu: ${df_curr["Sales"].sum():,.0f}')
print(f'Năm {PREV_YEAR}:   {len(df_prev):,} giao dịch | Doanh thu: ${df_prev["Sales"].sum():,.0f}')

Phạm vi dữ liệu sau dịch: 2023-01-03 → 2026-12-30
YTD 2026: 989 giao dịch | Doanh thu: $212,844
Năm 2025:   2,587 giao dịch | Doanh thu: $609,206


In [8]:
# ═══════════════════════════════════════════════════════════════
# 3. TRỌNG SỐ THÁNG: LẤY TỪ MODEL DỰ ĐOÁN (QUAN TRỌNG)
# ═══════════════════════════════════════════════════════════════
CUT_M = CUTOFF.month
CUT_D = CUTOFF.day
days_in_cut_month = calendar.monthrange(CUTOFF.year, CUT_M)[1]

with open('/content/pace_weight.json', encoding='utf-8') as f:
    _pw = json.load(f)
weights_raw = _pw['monthly_weights']          # dict {"1":0.082, "2":0.079, ...}
weights = {int(k): v for k, v in weights_raw.items()}

full_weight    = sum(weights[m] for m in range(1, CUT_M))
partial_weight = weights[CUT_M] * (CUT_D / days_in_cut_month)
PACE_WEIGHT    = full_weight + partial_weight

print(f'\nPACE WEIGHT CALCULATION:')
print(f'  Tháng hoàn chỉnh T1-T{CUT_M-1}: {full_weight:.4f} ({full_weight*100:.7f}%)')
print(f'  T{CUT_M} partial ({CUT_D}/{days_in_cut_month} ngày): {partial_weight:.4f} ({partial_weight*100:.6f}%)')
print(f'  ► PACE WEIGHT = {PACE_WEIGHT:.6f} = {PACE_WEIGHT*100:.3f}%')
print(f'  (Nếu tuyến tính: {(CUTOFF - YR_START).days/365*100:.6f}% — chênh {(PACE_WEIGHT-(CUTOFF-YR_START).days/365)*100:+.6f}% do mùa vụ)')


PACE WEIGHT CALCULATION:
  Tháng hoàn chỉnh T1-T5: 0.3101 (31.0100000%)
  T6 partial (5/30 ngày): 0.0110 (1.103333%)
  ► PACE WEIGHT = 0.321133 = 32.113%
  (Nếu tuyến tính: 42.465753% — chênh -10.352420% do mùa vụ)


In [9]:
# ═══════════════════════════════════════════════════════════════
# 4. TÍNH MỤC TIÊU NĂM & MỤC TIÊU THỜI ĐIỂM HIỆN TẠI
# ═══════════════════════════════════════════════════════════════
total_prev = df_prev['Sales'].sum()

annual_goals = {
    'total_revenue':       round(total_prev * GROWTH['total_revenue'], 0),
    'furniture':           round(df_prev[df_prev['Category']=='Furniture']['Sales'].sum() * GROWTH['furniture'], 0),
    'office_supplies':     round(df_prev[df_prev['Category']=='Office Supplies']['Sales'].sum() * GROWTH['office_supplies'], 0),
    'technology':          round(df_prev[df_prev['Category']=='Technology']['Sales'].sum() * GROWTH['technology'], 0),
    'total_profit':        round(df_prev['Profit'].sum() * GROWTH['total_profit'], 0),
    'consumer_segment':    round(df_prev[df_prev['Segment']=='Consumer']['Sales'].sum() * GROWTH['consumer_segment'], 0),
    'corporate_segment':   round(df_prev[df_prev['Segment']=='Corporate']['Sales'].sum() * GROWTH['corporate_segment'], 0),
    'home_office_segment': round(df_prev[df_prev['Segment']=='Home Office']['Sales'].sum() * GROWTH['home_office_segment'], 0),
}

# Mục tiêu thời điểm hiện tại = annual_goal × pace_weight (CÔNG THỨC CHUẨN)
pace_goals = {k: round(v * PACE_WEIGHT, 0) for k, v in annual_goals.items()}

print('Mục tiêu năm & mục tiêu hiện tại (điều chỉnh mùa vụ):')
print(f'  {"Chỉ số":<25} {"Mục tiêu năm":>12} {"KPI hiện tại":>12} {"Tỷ lệ":>8}')
print('  ' + '-'*60)
for k in annual_goals:
    ag = annual_goals[k]; pg = pace_goals[k]
    print(f'  {k:<25} {ag:>12,.0f} {pg:>12,.0f} {PACE_WEIGHT*100:>7.2f}%')

Mục tiêu năm & mục tiêu hiện tại (điều chỉnh mùa vụ):
  Chỉ số                    Mục tiêu năm KPI hiện tại    Tỷ lệ
  ------------------------------------------------------------
  total_revenue                  700,586      224,982   32.11%
  furniture                      218,792       70,261   32.11%
  office_supplies                206,013       66,158   32.11%
  technology                     267,110       85,778   32.11%
  total_profit                    98,154       31,521   32.11%
  consumer_segment               332,488      106,773   32.11%
  corporate_segment              238,172       76,485   32.11%
  home_office_segment            115,759       37,174   32.11%


In [10]:
# ═══════════════════════════════════════════════════════════════
# 5. XÂY DỰNG PARAMETER_CHART.JSON
# ═══════════════════════════════════════════════════════════════

df_curr['Month']   = df_curr['Order Date'].dt.month
df_prev['Month']   = df_prev['Order Date'].dt.month
df_curr['Quarter'] = df_curr['Order Date'].dt.quarter
df_prev['Quarter'] = df_prev['Order Date'].dt.quarter

# --- KPI CARDS ---
total_curr    = df_curr['Sales'].sum()
profit_curr   = df_curr['Profit'].sum()
margin_curr   = profit_curr / total_curr
orders_curr   = len(df_curr)
aov_curr      = total_curr / orders_curr

prev_ytd_sales  = df_prev_ytd['Sales'].sum()
prev_ytd_profit = df_prev_ytd['Profit'].sum()
prev_ytd_orders = len(df_prev_ytd)

def pct_chg(curr, prev): return round((curr-prev)/prev*100, 1) if prev else 0

kpi_cards = [
    {'id':'revenue',  'label':'Doanh thu YTD',    'value':round(total_curr,0),
     'prev_value':round(prev_ytd_sales,0), 'annual_goal':int(annual_goals['total_revenue']),
     'pace_goal':int(pace_goals['total_revenue']), 'yoy':pct_chg(total_curr,prev_ytd_sales), 'format':'currency'},
    {'id':'profit',   'label':'Lợi nhuận YTD',    'value':round(profit_curr,0),
     'prev_value':round(prev_ytd_profit,0), 'annual_goal':int(annual_goals['total_profit']),
     'pace_goal':int(pace_goals['total_profit']), 'yoy':pct_chg(profit_curr,prev_ytd_profit), 'format':'currency'},
    {'id':'margin',   'label':'Biên lợi nhuận',   'value':round(margin_curr*100,1),
     'prev_value':round(prev_ytd_profit/prev_ytd_sales*100,1) if prev_ytd_sales else 0,
     'annual_goal':PROFIT_MARGIN_TARGET*100, 'pace_goal':PROFIT_MARGIN_TARGET*100,
     'yoy':pct_chg(margin_curr,prev_ytd_profit/prev_ytd_sales if prev_ytd_sales else margin_curr), 'format':'percent'},
    {'id':'orders',   'label':'Số giao dịch',      'value':orders_curr,
     'prev_value':prev_ytd_orders, 'annual_goal':None, 'pace_goal':None,
     'yoy':pct_chg(orders_curr,prev_ytd_orders), 'format':'number'},
    {'id':'aov',      'label':'Giá trị đơn TB',   'value':round(aov_curr,0),
     'prev_value':round(prev_ytd_sales/prev_ytd_orders,0) if prev_ytd_orders else 0,
     'annual_goal':int(AOV_TARGET), 'pace_goal':int(AOV_TARGET),
     'yoy':pct_chg(aov_curr,prev_ytd_sales/prev_ytd_orders if prev_ytd_orders else aov_curr), 'format':'currency'},
    {'id':'discount', 'label':'Chiết khấu TB',    'value':round(df_curr['Discount'].mean()*100,1),
     'prev_value':round(df_prev_ytd['Discount'].mean()*100,1),
     'annual_goal':None, 'pace_goal':None,
     'yoy':pct_chg(df_curr['Discount'].mean(), df_prev_ytd['Discount'].mean()), 'format':'percent'},
]

print('KPI cards:')
for c in kpi_cards:
    pg_str = f"pace={c['pace_goal']:,.0f}" if c['pace_goal'] else 'no target'
    print(f"  {c['label']}: value={c['value']}  {pg_str}  yoy={c['yoy']:+.1f}%")

KPI cards:
  Doanh thu YTD: value=212844.0  pace=224,982  yoy=+10.8%
  Lợi nhuận YTD: value=32627.0  pace=31,521  yoy=+40.7%
  Biên lợi nhuận: value=15.3  pace=14  yoy=+27.0%
  Số giao dịch: value=989  no target  yoy=+31.0%
  Giá trị đơn TB: value=215.0  pace=800  yoy=-15.5%
  Chiết khấu TB: value=16.2  no target  yoy=-0.9%


In [11]:
# --- PACE CHART ---
cat_curr = df_curr.groupby('Category')['Sales'].sum()
seg_curr = df_curr.groupby('Segment')['Sales'].sum()

def make_pace(label, actual, annual_g, pace_g):
    pct_actual = round(actual/annual_g*100, 1)
    pct_pace   = round(pace_g/annual_g*100, 1)
    delta      = round(pct_actual - pct_pace, 1)
    status = 'ahead' if delta >= 1 else ('on_pace' if delta >= -1 else 'behind')
    return {'label':label,'actual':round(actual,0),'annual_goal':annual_g,'pace_goal':pace_g,
            'pct_of_goal':pct_actual,'pct_pace':pct_pace,'delta':delta,'status':status}

pace_chart = [
    make_pace('Tổng doanh thu',  total_curr,                           annual_goals['total_revenue'],       pace_goals['total_revenue']),
    make_pace('Technology',      cat_curr.get('Technology',0),         annual_goals['technology'],          pace_goals['technology']),
    make_pace('Furniture',       cat_curr.get('Furniture',0),          annual_goals['furniture'],           pace_goals['furniture']),
    make_pace('Office Supplies', cat_curr.get('Office Supplies',0),    annual_goals['office_supplies'],     pace_goals['office_supplies']),
    make_pace('Consumer',        seg_curr.get('Consumer',0),           annual_goals['consumer_segment'],    pace_goals['consumer_segment']),
    make_pace('Corporate',       seg_curr.get('Corporate',0),          annual_goals['corporate_segment'],   pace_goals['corporate_segment']),
    make_pace('Home Office',     seg_curr.get('Home Office',0),        annual_goals['home_office_segment'], pace_goals['home_office_segment']),
]

print('PACE CHART — so sánh thực tế vs kỳ vọng mùa vụ:')
print(f'  {"Danh mục":<20} {"Thực tế":>10} {"KPI h/tại":>10} {"KH năm":>10} {"% đạt":>7} {"% kỳ vọng":>10} {"Δ":>6} {"Trạng thái"}')
print('  ' + '-'*90)
for p in pace_chart:
    sym = '✅ Vượt  ' if p['status']=='ahead' else ('🟡 Đúng  ' if p['status']=='on_pace' else '🔴 Chậm  ')
    print(f'  {p["label"]:<20} {p["actual"]:>10,.0f} {p["pace_goal"]:>10,.0f} {p["annual_goal"]:>10,.0f} {p["pct_of_goal"]:>6.1f}% {p["pct_pace"]:>9.1f}% {p["delta"]:>+5.1f}pt  {sym}')

PACE CHART — so sánh thực tế vs kỳ vọng mùa vụ:
  Danh mục                Thực tế  KPI h/tại     KH năm   % đạt  % kỳ vọng      Δ Trạng thái
  ------------------------------------------------------------------------------------------
  Tổng doanh thu          212,844    224,982    700,586   30.4%      32.1%  -1.7pt  🔴 Chậm  
  Technology               85,010     85,778    267,110   31.8%      32.1%  -0.3pt  🟡 Đúng  
  Furniture                52,397     70,261    218,792   23.9%      32.1%  -8.2pt  🔴 Chậm  
  Office Supplies          75,438     66,158    206,013   36.6%      32.1%  +4.5pt  ✅ Vượt  
  Consumer                 96,191    106,773    332,488   28.9%      32.1%  -3.2pt  🔴 Chậm  
  Corporate                72,177     76,485    238,172   30.3%      32.1%  -1.8pt  🔴 Chậm  
  Home Office              44,477     37,174    115,759   38.4%      32.1%  +6.3pt  ✅ Vượt  


In [12]:
# --- MONTHLY TREND ---
mn_curr = df_curr.groupby('Month').agg(Sales=('Sales','sum'),Profit=('Profit','sum')).reindex(range(1,13),fill_value=0)
mn_prev = df_prev.groupby('Month').agg(Sales=('Sales','sum'),Profit=('Profit','sum')).reindex(range(1,13),fill_value=0)

# Mục tiêu từng tháng (để vẽ đường target trên trend chart)
monthly_targets = [round(annual_goals['total_revenue'] * float(weights[m]), 0) for m in range(1,13)]

monthly_trend = {
    'months':          ['T1','T2','T3','T4','T5','T6','T7','T8','T9','T10','T11','T12'],
    'sales_curr':      [round(float(mn_curr.loc[m,'Sales']),0)  for m in range(1,13)],
    'sales_prev':      [round(float(mn_prev.loc[m,'Sales']),0)  for m in range(1,13)],
    'profit_curr':     [round(float(mn_curr.loc[m,'Profit']),0) for m in range(1,13)],
    'profit_prev':     [round(float(mn_prev.loc[m,'Profit']),0) for m in range(1,13)],
    'monthly_targets': monthly_targets,
    'completed_months': int(CUT_M - 1),   # T1-T5 hoàn chỉnh
    'partial_month':    int(CUT_M),        # T6 đang chạy
    'curr_year':        int(CUTOFF.year),
    'prev_year':        int(PREV_YEAR),
}

print(f'Monthly trend: {monthly_trend["curr_year"]} vs {monthly_trend["prev_year"]}')
print(f'  Tháng hoàn chỉnh: T1-T{monthly_trend["completed_months"]}')
print(f'  Tháng đang chạy: T{monthly_trend["partial_month"]}')

Monthly trend: 2026 vs 2025
  Tháng hoàn chỉnh: T1-T5
  Tháng đang chạy: T6


In [13]:
# --- QUARTERLY ---
q_month_map = {1:[1,2,3], 2:[4,5,6], 3:[7,8,9], 4:[10,11,12]}
q_curr = df_curr.groupby('Quarter').agg(Sales=('Sales','sum'),Profit=('Profit','sum')).reindex([1,2,3,4],fill_value=0)
q_prev = df_prev.groupby('Quarter').agg(Sales=('Sales','sum'),Profit=('Profit','sum')).reindex([1,2,3,4],fill_value=0)

quarterly = []
for q in [1,2,3,4]:
    q_w       = sum(float(weights[m]) for m in q_month_map[q])
    q_annual  = round(annual_goals['total_revenue'] * q_w, 0)
    actual    = round(float(q_curr.loc[q,'Sales']), 0)
    has_data  = actual > 0

    # Pace goal trong quý: chỉ tính cho quý đang chạy (Q2)
    if q < CUT_M//3 + 1 and has_data:          # Q1: hoàn chỉnh
        q_pace = q_annual
    elif q == (CUT_M - 1)//3 + 1 and has_data: # Quý hiện tại
        months_done = [m for m in q_month_map[q] if m < CUT_M]
        months_partial = CUT_M if CUT_M in q_month_map[q] else None
        pw = sum(float(weights[m]) for m in months_done)
        if months_partial: pw += float(weights[months_partial]) * (CUT_D/days_in_cut_month)
        q_pace = round(annual_goals['total_revenue'] * pw, 0)
    else:
        q_pace = 0

    delta_pace = round((actual-q_pace)/q_pace*100,1) if q_pace > 0 and has_data else None
    delta_prev = round((actual-float(q_prev.loc[q,'Sales']))/float(q_prev.loc[q,'Sales'])*100,1) if q_prev.loc[q,'Sales'] > 0 and has_data else None

    quarterly.append({
        'quarter':f'Q{q}','actual':actual,'prev':round(float(q_prev.loc[q,'Sales']),0),
        'annual_goal':q_annual,'pace_goal':q_pace,
        'delta_vs_pace':delta_pace,'delta_vs_prev':delta_prev,'has_data':has_data,
    })

print('Quarterly performance:')
for q in quarterly:
    dp = f'{q["delta_vs_pace"]:+.1f}% vs KPI' if q['delta_vs_pace'] is not None else 'pending'
    print(f'  {q["quarter"]}: actual={q["actual"]:>9,.0f}  pace={q["pace_goal"]:>9,.0f}  {dp}')

Quarterly performance:
  Q1: actual=  123,145  pace=  107,190  +14.9% vs KPI
  Q2: actual=   89,699  pace=  156,441  -42.7% vs KPI
  Q3: actual=        0  pace=        0  pending
  Q4: actual=        0  pace=        0  pending


In [14]:
# --- CATEGORY MIX, REGION, SUB-CATEGORY ---

# Category mix
cat_colors = {'Technology':'#2563EB','Office Supplies':'#0EA5E9','Furniture':'#64748B'}
cat_df = df_curr.groupby('Category').agg(Sales=('Sales','sum'),Profit=('Profit','sum')).reset_index()
cat_prev = df_prev_ytd.groupby('Category')['Sales'].sum().rename('prev')
cat_df = cat_df.merge(cat_prev, on='Category', how='left').fillna(0)
cat_df['margin'] = (cat_df['Profit']/cat_df['Sales']*100).round(1)
cat_df['yoy']    = ((cat_df['Sales']-cat_df['prev'])/cat_df['prev']*100).round(1)
category_mix = [{'label':r['Category'],'value':round(float(r['Sales']),0),
                  'profit':round(float(r['Profit']),0),'margin':float(r['margin']),
                  'yoy':float(r['yoy']),'color':cat_colors.get(r['Category'],'#94A3B8')}
                 for _,r in cat_df.iterrows()]

# Region performance
reg_df   = df_curr.groupby('Region').agg(Sales=('Sales','sum'),Profit=('Profit','sum'),Txn=('Sales','count')).reset_index()
reg_prev = df_prev_ytd.groupby('Region')['Sales'].sum().rename('prev')
reg_df   = reg_df.merge(reg_prev, on='Region', how='left').fillna(0)
reg_df['margin'] = (reg_df['Profit']/reg_df['Sales']*100).round(1)
reg_df['yoy']    = ((reg_df['Sales']-reg_df['prev'])/reg_df['prev']*100).round(1)
region_perf = [{'region':r['Region'],'sales':round(float(r['Sales']),0),'profit':round(float(r['Profit']),0),
                 'txn':int(r['Txn']),'margin':float(r['margin']),'yoy':float(r['yoy'])}
                for _,r in reg_df.sort_values('Sales',ascending=False).iterrows()]

# Top sub-categories
sc_df   = df_curr.groupby('Sub-Category').agg(Sales=('Sales','sum'),Profit=('Profit','sum')).sort_values('Sales',ascending=False).head(10).reset_index()
sc_prev = df_prev_ytd.groupby('Sub-Category')['Sales'].sum().rename('prev')
sc_df   = sc_df.merge(sc_prev, on='Sub-Category', how='left').fillna(0)
sc_df['margin'] = (sc_df['Profit']/sc_df['Sales']*100).round(1)
sc_df['yoy']    = ((sc_df['Sales']-sc_df['prev'])/sc_df['prev']*100).round(1)
top_subcats = [{'label':r['Sub-Category'],'sales':round(float(r['Sales']),0),'profit':round(float(r['Profit']),0),
                 'margin':float(r['margin']),'yoy':float(r['yoy'])}
                for _,r in sc_df.iterrows()]

print('Category mix:', [(c['label'], f"${c['value']:,.0f}", f"{c['yoy']:+.1f}%") for c in category_mix])
print('Regions:', [(r['region'], f"${r['sales']:,.0f}", f"{r['margin']}%") for r in region_perf])

Category mix: [('Furniture', '$52,397', '-2.2%'), ('Office Supplies', '$75,438', '+39.5%'), ('Technology', '$85,010', '+0.5%')]
Regions: [('West', '$81,355', '18.8%'), ('Central', '$60,555', '12.0%'), ('East', '$35,473', '11.1%'), ('South', '$35,461', '17.4%')]


In [15]:
# --- BULLET POINTS (key findings) ---
p0    = pace_chart[0]
best  = max(pace_chart[1:], key=lambda x: x['delta'])
worst = min(pace_chart[1:], key=lambda x: x['delta'])
margin_ok = margin_curr >= PROFIT_MARGIN_TARGET

bullets = [
    {'icon':'📈' if p0['delta']>=0 else '⚠️',
     'text':f"Doanh thu YTD {'vượt' if p0['delta']>=0 else 'chậm'} tiến độ mùa vụ {abs(p0['delta']):.1f} điểm % — {'+' if kpi_cards[0]['yoy']>=0 else ''}{kpi_cards[0]['yoy']:.1f}% so cùng kỳ {PREV_YEAR}",
     'status':'positive' if p0['delta']>=0 else 'warning'},
    {'icon':'🏆',
     'text':f"{best['label']} dẫn đầu: {best['pct_of_goal']:.1f}% kế hoạch năm, vượt tiến độ {best['delta']:+.1f} điểm % so kỳ vọng mùa vụ",
     'status':'positive'},
    {'icon':'🔴' if worst['delta'] < -3 else '⚠️',
     'text':f"{worst['label']} cần chú ý: {worst['pct_of_goal']:.1f}% kế hoạch, {'chậm' if worst['delta']<0 else 'đúng'} tiến độ {worst['delta']:.1f} điểm % — cần đẩy mạnh Q3",
     'status':'warning'},
    {'icon':'✅' if margin_ok else '💰',
     'text':f"Biên LN {margin_curr*100:.1f}% — {'vượt' if margin_ok else 'chưa đạt'} mục tiêu {PROFIT_MARGIN_TARGET*100:.0f}%  |  Giá trị đơn TB ${aov_curr:.0f} ({'✓' if aov_curr>=AOV_TARGET else '↓'} KPI ${AOV_TARGET:.0f})",
     'status':'positive' if margin_ok else 'neutral'},
]

print('Key findings:')
for b in bullets: print(f'  {b["icon"]} {b["text"]}')

Key findings:
  ⚠️ Doanh thu YTD chậm tiến độ mùa vụ 1.7 điểm % — +10.8% so cùng kỳ 2025
  🏆 Home Office dẫn đầu: 38.4% kế hoạch năm, vượt tiến độ +6.3 điểm % so kỳ vọng mùa vụ
  🔴 Furniture cần chú ý: 23.9% kế hoạch, chậm tiến độ -8.2 điểm % — cần đẩy mạnh Q3
  ✅ Biên LN 15.3% — vượt mục tiêu 14%  |  Giá trị đơn TB $215 (↓ KPI $800)


In [16]:
# ═══════════════════════════════════════════════════════════════
# 6. LƯU FILE ĐẦU RA
# ═══════════════════════════════════════════════════════════════

# --- parameter_chart.json ---
parameter_chart = {
    '_meta': {
        'generated': CUTOFF.strftime('%Y-%m-%d'),
        'curr_year': int(CUTOFF.year), 'prev_year': int(PREV_YEAR),
        'pace_weight': round(PACE_WEIGHT, 6),
        'pace_weight_pct': round(PACE_WEIGHT*100, 3),
        'pace_formula': f'Σ weight(T1-T{CUT_M-1}) + weight(T{CUT_M}) × ({CUT_D}/{days_in_cut_month})',
    },
    'kpi_cards':        kpi_cards,
    'bullet_points':    bullets,
    'pace_chart':       pace_chart,
    'monthly_trend':    monthly_trend,
    'category_mix':     category_mix,
    'region_performance': region_perf,
    'top_subcategories': top_subcats,
    'quarterly':        quarterly,
}
with open('parameter_chart.json', 'w', encoding='utf-8') as f:
    json.dump(parameter_chart, f, ensure_ascii=False, indent=2)

# --- kpi.json ---
kpi_out = {
    '_note': 'Chỉnh growth_targets → chạy lại notebook → dashboard tự cập nhật',
    'year': int(CUTOFF.year), 'cutoff_date': CUTOFF.strftime('%Y-%m-%d'),
    'growth_targets': GROWTH,
    'annual_goals': {k:int(v) for k,v in annual_goals.items()},
    'pace_goals':   {k:int(v) for k,v in pace_goals.items()},
    'monthly_weights_2025': {str(m):round(float(weights[m]),6) for m in range(1,13)},
    'pace_weight': round(PACE_WEIGHT, 6),
    'pace_weight_pct': round(PACE_WEIGHT*100, 3),
    'formula': 'pace_goal = annual_goal × [Σ weight(T1..T_prev) + weight(T_cur) × day/days_in_month]',
    'profit_margin_target': PROFIT_MARGIN_TARGET,
    'avg_order_value_target': AOV_TARGET,
}
with open('kpi.json', 'w', encoding='utf-8') as f:
    json.dump(kpi_out, f, ensure_ascii=False, indent=2)

print('✅ parameter_chart.json saved')
print('✅ kpi.json saved')
print()
print('═'*55)
print(f'  TỔNG KẾT — {CUTOFF.strftime("%d/%m/%Y")}')
print('═'*55)
print(f'  Doanh thu YTD:      ${total_curr:>12,.0f}')
print(f'  KPI thời điểm hiện tại: ${pace_goals["total_revenue"]:>10,.0f}  ({PACE_WEIGHT*100:.2f}% x ${annual_goals["total_revenue"]:,.0f})')
print(f'  vs KPI:             {(total_curr-pace_goals["total_revenue"])/pace_goals["total_revenue"]*100:>+11.2f}%')
print(f'  Lợi nhuận YTD:      ${profit_curr:>12,.0f}')
print(f'  Biên lợi nhuận:     {margin_curr*100:>11.1f}%  (target: {PROFIT_MARGIN_TARGET*100:.0f}%)')
print('═'*55)
print('  → Mở dashboard.html để xem visualization')

✅ parameter_chart.json saved
✅ kpi.json saved

═══════════════════════════════════════════════════════
  TỔNG KẾT — 05/06/2026
═══════════════════════════════════════════════════════
  Doanh thu YTD:      $     212,844
  KPI thời điểm hiện tại: $   224,982  (32.11% x $700,586)
  vs KPI:                   -5.40%
  Lợi nhuận YTD:      $      32,627
  Biên lợi nhuận:            15.3%  (target: 14%)
═══════════════════════════════════════════════════════
  → Mở dashboard.html để xem visualization
